# 04 · RQ2 – Expertise level of exposure, effectiveness and its determinants (Section 4.2)

Produces Figures 11–26 and Tables 4–8 in the order of Section 4.2, plus the Appendix D items that support it. Requires `01_constructs_and_checks.ipynb`.

`a` = left panel, `b` = right panel as placed in the thesis. Main-text items go to `outputs/3_ch4_2_rq2_expertise/`, appendix items to `outputs/4_appendix_D/`.

| Section | Thesis item | Output |
|---|---|---|
| 4.2.1 | Figure 11 – coverage by ridge expertise, R5 | `fig11a_coverage_by_ridge_claude_ai_R5.png`, `fig11b_..._1p_api_R5.png` |
| 4.2.1 | Figure 12 – net skill-coverage gap | `fig12_net_skill_coverage_gap.png` (+ `fig12_data.csv`) |
| 4.2.2 | Figure 13 – ridge expertise vs subprocess exposure, R5 | `fig13a_..._gateON.png`, `fig13b_..._gateOFF.png` |
| 4.2.2 | Figure 14 – concentration by ridge expertise, R5 | `fig14a_..._gateON.png`, `fig14b_..._gateOFF.png` |
| 4.2.2 | Figure 15 – ridge expertise vs task exposure, R5 | `fig15a/b_...png` |
| 4.2.2 | Figure 16 – task dispersion by ridge expertise, R5 | `fig16a/b_...png` |
| 4.2.2 | Expertise level of the entire process, its concentration and its exposure | `text_values_4_2_2.csv` |
| 4.2.2 | Figure 17 – concentration by subprocess value, R5 | `fig17a/b_...png` |
| 4.2.2 | Figure 18 – task dispersion by task value, R5 | `fig18a/b_...png` |
| 4.2.3 | Figure 19 – effective exposure vs task coverage (a = R4, b = R5) | `fig19a_..._R4.png`, `fig19b_..._R5.png` |
| 4.2.3 | Table 4 – effective exposure − task coverage | `table04_effective_minus_coverage.csv` |
| 4.2.3 | Figure 20 – effective vs observed exposure, R5 | `fig20a/b_...png` |
| 4.2.3 | Table 5 – effective − observed exposure | `table05_effective_minus_observed.csv` |
| 4.2.3 | Table D7 + Figure D5 – task-level decomposition of the coverage gap | `4_appendix_D/tableD7_coverage_gap_decomposition.csv`, `figD5_task_contribution_R5.png` |
| 4.2.4 | Figure 21 – success by ridge expertise (a = R4, b = R5) | `fig21a/b_...png` |
| 4.2.4 | Table 6 – primitives R4 vs R5 | `table06_primitives.csv` (+ `table06_use_case_diagnostic.csv`) |
| 4.2.4 | Table D2 – success by subprocess, platform, release | `4_appendix_D/tableD2_success_by_subprocess.csv` |
| 4.2.4 | Figures 22–23 – human-only time / autonomy, R5 | `fig22a/b`, `fig23a/b` |
| 4.2.4 | Table 7 – Claude.ai success regressions (pooled) | `table07_claude_success_regression.csv` |
| 4.2.4 | Figures 24–25 – context length, 1P API, R5 | `fig24_context_vs_success_R5.png`, `fig25_context_vs_ridge_R5.png` |
| 4.2.4 | Figure D4 – context length vs success, R4 | `4_appendix_D/figD4_context_vs_success_R4.png` |
| 4.2.4 | Figure D2 – input vs output index, R5 | `4_appendix_D/figD2_input_vs_output_index_R5.png` |
| 4.2.4 | Table 8 – 1P API success regressions (pooled) | `table08_api_success_regression.csv` |
| 4.2.4 | Table D1 – Table 8 by release | `4_appendix_D/tableD1_api_success_regression_by_release.csv` |
| 4.2.4 | Table D8 – robustness: task-clustered SEs, release indicator | `4_appendix_D/tableD8a_claude_success_robustness.csv`, `tableD8b_api_success_robustness.csv` |
| 4.2.4 | Table D10 – Table 7 by release | `4_appendix_D/tableD10_claude_success_regression_by_release.csv` |
| 4.2.4 | Figure 26 – usage share vs cost index, R5 | `fig26a_cost_vs_share_R5.png`, `fig26b_cost_vs_share_partial_R5.png` |
| all | Slopes / R² quoted in the text | `fit_statistics.csv` |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import r2r_common as rc

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
tm, panel = rc.load_data()
task_exp = rc.read_construct("task_exposure.csv")
sub_exp = rc.read_construct("subprocess_exposure.csv")
proc_exp = rc.read_construct("process_exposure.csv")
task_disp = rc.read_construct("task_dispersion.csv")
sub_conc = rc.read_construct("subprocess_concentration.csv")
proc_conc = rc.read_construct("process_concentration.csv")
eff = rc.read_construct("effective_exposure.csv")
r2r_idx = rc.read_construct("r2r_indices.csv")

t = tm.set_index("task_id")
ridge = rc.get_ridge_task_years(tm)
gate_tag = lambda g: "gateON" if g else "gateOFF"
gate_txt = lambda g: "gate ON" if g else "gate OFF"
panels = [(r, g) for r in rc.EXPOSURE_RELEASES for g in rc.GATE_STATES]

# time-fraction-weighted subprocess (and entire-process) ridge expertise and value
sub_attr = pd.DataFrame([{"subprocess": sc,
                          "ridge_years": rc._weighted_mean(t.loc[rc.scope_task_ids(tm, sc), "ridge_task_years"],
                                                           t.loc[rc.scope_task_ids(tm, sc), "task_time_fraction"]),
                          "value_usd": rc._weighted_mean(t.loc[rc.scope_task_ids(tm, sc), "wage_hourly_mean"],
                                                         t.loc[rc.scope_task_ids(tm, sc), "task_time_fraction"])}
                         for sc in rc.SUBPROCESS_ORDER + ["Entire Process"]]).set_index("subprocess")
sub_attr.round(2)

[load] task_master: 212 rows, 91 in scope ({'Master Data Management': 8, 'Record': 31, 'Close & Consolidation': 5, 'Report': 13, 'Analysis & Business Insight': 12, 'Planning & Budgeting': 22})
[load] panel: 9,522 rows, releases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)], 50 tasks; rows whose task is not in scope: 0


,ridge_years,value_usd
subprocess,,
Master Data Management,14.89,66.25
Record,13.05,29.17
Close & Consolidation,13.65,33.88
Report,14.40,49.47
Analysis & Business Insight,15.19,65.52
Planning & Budgeting,15.35,59.40
Entire Process,14.28,47.61


## 4.2.1 Expertise composition of task coverage

### Figure 11 – Time-weighted distribution of ridge expertise: all in-scope tasks vs covered tasks, R5 (a Claude.ai, b 1P API)

In [2]:
for p, letter in [("claude_ai", "a"), ("1p_api", "b")]:
    covered = sorted(rc.get_covered_task_ids(panel, 5, p) & set(t.index))
    r_all, w_all = t["ridge_task_years"], t["task_time_fraction"]
    r_cov, w_cov = r_all.loc[covered], w_all.loc[covered]
    mean_all, mean_cov = rc._weighted_mean(r_all, w_all), rc._weighted_mean(r_cov, w_cov)
    bins = np.linspace(r_all.min(), r_all.max(), 26)
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    ax.hist(r_all, bins=bins, weights=w_all, density=True, color="#4B90DD", alpha=0.55, edgecolor="white", linewidth=0.4, zorder=2)
    ax.hist(r_cov, bins=bins, weights=w_cov, density=True, color="#F88F49", alpha=0.6, edgecolor="white", linewidth=0.4, zorder=3)
    ax.axvline(mean_all, color="#4B90DD", linestyle="--", linewidth=1.8, zorder=4)
    ax.axvline(mean_cov, color="#F88F49", linestyle="--", linewidth=1.8, zorder=4)
    y_txt = ax.get_ylim()[1] * 0.97
    ax.annotate(f"Mean (all): {mean_all:.1f} yrs", (mean_all, y_txt), xytext=(6, 0), textcoords="offset points",
                va="top", color="#4B90DD", fontsize=9, fontweight="semibold")
    ax.annotate(f"Mean (covered): {mean_cov:.1f} yrs", (mean_cov, y_txt * 0.9), xytext=(6, 0), textcoords="offset points",
                va="top", color="#F88F49", fontsize=9, fontweight="semibold")
    rc.clean_axes(ax)
    ax.set_xlabel("Ridge Expertise (predicted task complexity, yrs of education)")
    ax.set_ylabel("Density (task-time-fraction weighted)")
    ax.set_title(f"Task coverage by ridge expertise: all tasks vs. {rc.PLATFORM_LABELS[p]}-covered -- Release 5", loc="left", fontsize=13)
    rc.legend_box(ax, [("#4B90DD", f"All in-scope tasks (n={len(r_all)})", "s"),
                       ("#F88F49", f"{rc.PLATFORM_LABELS[p]}-covered tasks (n={len(covered)})", "s")], loc="upper left")
    fig.tight_layout()
    rc.save_figure(fig, "rq2", f"fig11{letter}_coverage_by_ridge_{p}_R5.png")
    print(f"{rc.PLATFORM_LABELS[p]} R5: mean all {mean_all:.2f}, covered {mean_cov:.2f}, gap {mean_all - mean_cov:+.2f} yrs")

findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/3_ch4_2_rq2_expertise/fig11a_coverage_by_ridge_claude_ai_R5.png
Claude.ai R5: mean all 14.28, covered 14.55, gap -0.27 yrs


[write] outputs/3_ch4_2_rq2_expertise/fig11b_coverage_by_ridge_1p_api_R5.png
1P API R5: mean all 14.28, covered 14.18, gap +0.10 yrs


### Figure 12 – Net skill-coverage gap over time, by subprocess and platform

In [3]:
gap = rc.compute_net_skill_coverage_gap(tm, panel)
rc.save_csv(gap, "rq2", "fig12_data.csv")
releases = list(rc.ANALYSIS_RELEASES)
mosaic = [["ALL", "ALL", "MDM", "REC", "CNC"], ["ALL", "ALL", "REP", "ABI", "PNB"]]
sp_key = {"MDM": "Master Data Management", "REC": "Record", "CNC": "Close & Consolidation",
          "REP": "Report", "ABI": "Analysis & Business Insight", "PNB": "Planning & Budgeting"}
lo, hi = gap["gap"].min(), gap["gap"].max()
ylim12 = (lo - (hi - lo) * 0.05, hi + (hi - lo) * 0.05)
fig, axd = plt.subplot_mosaic(mosaic, figsize=(17, 7.5))
for key, sp in [("ALL", "Entire Process")] + list(sp_key.items()):
    ax = axd[key]
    big = key == "ALL"
    for p in rc.PLATFORMS:
        s = gap[(gap["subprocess"] == sp) & (gap["platform"] == p)].sort_values("release")
        ax.plot(s["release"], s["gap"], color=rc.PLATFORM_COLORS[p], marker=rc.PLATFORM_MARKERS[p],
                linewidth=2.6 if big else 1.8, markersize=6.5 if big else 4.5, markeredgecolor="white", markeredgewidth=0.7)
    ax.axhline(0, color=rc.BASELINE_COLOR, linewidth=rc.BASELINE_LINEWIDTH, linestyle="--", zorder=1)
    ax.set_title(sp, fontsize=13 if big else 9.5)
    ax.set_ylim(*ylim12)
    rc.release_ticks(ax, releases, short=not big)
    ax.tick_params(labelsize=9.5 if big else 7.5)
    ax.set_ylabel("Mean ridge-expertise gap (all - covered, yrs)" if big else "Gap (yrs)", fontsize=11 if big else 8,
                  labelpad=14 if big else 6)
    rc.clean_axes(ax)
rc.legend_box(fig, [(rc.PLATFORM_COLORS[p], rc.PLATFORM_LABELS[p], rc.PLATFORM_MARKERS[p]) for p in rc.PLATFORMS],
              loc="lower center", bbox_to_anchor=(0.5, -0.02), ncol=2)
fig.suptitle("Net skill-coverage gap over time, by subprocess (mean ridge expertise of all in-scope tasks minus "
             "mean ridge expertise of covered tasks)", fontsize=13, fontweight="bold")
fig.tight_layout(rect=[0, 0.04, 1, 0.92])
rc.save_figure(fig, "rq2", "fig12_net_skill_coverage_gap.png")
gap.pivot_table(index=["subprocess", "platform"], columns="release", values="gap").round(2)

[write] outputs/3_ch4_2_rq2_expertise/fig12_data.csv


[write] outputs/3_ch4_2_rq2_expertise/fig12_net_skill_coverage_gap.png


release                                   1     2     3     4     5
subprocess                  platform                               
Analysis & Business Insight 1p_api      NaN   NaN  0.11  0.02  0.11
                            claude_ai -0.07 -0.15 -0.25 -0.25 -0.25
Close & Consolidation       1p_api      NaN   NaN  0.75  0.75  0.75
                            claude_ai  0.75  0.75  0.75  0.75  0.75
Entire Process              1p_api      NaN   NaN -0.19 -0.29  0.10
                            claude_ai -0.29 -0.23 -0.40 -0.13 -0.27
Master Data Management      1p_api      NaN   NaN  0.35  0.21  0.21
                            claude_ai  0.20  0.20  0.20  0.21 -0.01
Planning & Budgeting        1p_api      NaN   NaN -0.35 -0.34 -0.99
                            claude_ai -0.65 -0.16 -0.55 -0.49 -0.42
Record                      1p_api      NaN   NaN -0.58 -0.82 -0.28
                            claude_ai -0.42 -0.27 -0.52 -0.22 -0.37
Report                      1p_api      NaN   NaN -0.74 -0.74 -0.74
                            claude_ai -0.80 -0.65 -0.74 -0.74 -0.80

## 4.2.2 Expertise composition of observed exposure and concentration

### Figure 13 – Ridge-estimated subprocess expertise vs subprocess observed exposure, R5 (a gate ON, b gate OFF)
Bubble area = conversation volume; the blue square is the entire process at (time-weighted process expertise, Rₒ,ₚ). The fit uses the six subprocesses only.

In [4]:
for g, letter in [(True, "a"), (False, "b")]:
    d = sub_exp[(sub_exp["release"] == 5) & (sub_exp["gate_on"] == g)].join(sub_attr, on="subprocess")
    pe_row = proc_exp[(proc_exp["release"] == 5) & (proc_exp["gate_on"] == g)].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 6.5))
    rc.clean_axes(ax)
    rc.subprocess_bubbles(ax, d, "ridge_years", "R_o")
    rc.draw_entire_process(ax, sub_attr.loc["Entire Process", "ridge_years"], pe_row["R_process"])
    xlim = (d["ridge_years"].min() - 0.15, d["ridge_years"].max() + 0.15)
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["ridge_years"], d["R_o"], xlim)
    rc.record_fit("Figure 13", f"R5 {gate_txt(g)}", "subprocess ridge expertise", "R_o", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.TEXT_BLACK, lab.replace("\n", ", "), "line")], loc="upper left")
    ax.set_xlim(*xlim)
    ax.set_ylim(0, max(1.0, d["R_o"].max() * 1.2))
    ax.set_xlabel("Ridge Expertise (yrs, subprocess mean)")
    ax.set_ylabel("Observed Exposure, R_o")
    ax.set_title(f"Ridge expertise vs. subprocess observed exposure -- Release 5, {gate_txt(g)}", loc="left", fontsize=12.5)
    fig.tight_layout()
    rc.save_figure(fig, "rq2", f"fig13{letter}_ridge_vs_subprocess_exposure_R5_{gate_tag(g)}.png")

findfont: Failed to find font weight semibold for Arial, now using 700.


findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/3_ch4_2_rq2_expertise/fig13a_ridge_vs_subprocess_exposure_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig13b_ridge_vs_subprocess_exposure_R5_gateOFF.png


### Figure 14 – Subprocess concentration by ridge expertise, R5 (a gate ON, b gate OFF)
The square sits at (time-weighted expertise of all in-scope tasks, process-level concentration), exactly as in Figure 7.

In [5]:
sc = sub_conc.join(sub_attr, on="subprocess")
xlim14, ylim14 = rc.global_limits([sc[(sc["release"] == r) & (sc["gate_on"] == g)] for r, g in panels], "ridge_years", "concentration")
ylim14 = (ylim14[0], max(ylim14[1], proc_conc["concentration"].max() * 1.15))
proc_x = sub_attr.loc["Entire Process", "ridge_years"]
xlim14 = (min(xlim14[0], proc_x - 0.2), max(xlim14[1], proc_x + 0.2))
for g, letter in [(True, "a"), (False, "b")]:
    d = sc[(sc["release"] == 5) & (sc["gate_on"] == g)]
    pc = proc_conc[(proc_conc["release"] == 5) & (proc_conc["gate_on"] == g)].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 6.5))
    rc.clean_axes(ax)
    rc.subprocess_bubbles(ax, d, "ridge_years", "concentration")
    rc.draw_entire_process(ax, proc_x, pc["concentration"])
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["ridge_years"], d["concentration"], xlim14)
    rc.record_fit("Figure 14", f"R5 {gate_txt(g)}", "subprocess ridge expertise", "concentration", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.MODE_COLOR_COOL, lab, "line")], loc="upper left")
    ax.set_xlim(*xlim14)
    ax.set_ylim(*ylim14)
    ax.set_xlabel("Subprocess ridge expertise (yrs)")
    ax.set_ylabel("Subprocess concentration")
    ax.set_title(f"Subprocess concentration by ridge expertise -- Release 5, {gate_txt(g)}")
    rc.save_figure(fig, "rq2", f"fig14{letter}_concentration_by_ridge_R5_{gate_tag(g)}.png")
    print(f"R5 {gate_txt(g)}: entire process at {proc_x:.2f} yrs, concentration {pc['concentration']:.3f}")

[write] outputs/3_ch4_2_rq2_expertise/fig14a_concentration_by_ridge_R5_gateON.png
R5 gate ON: entire process at 14.28 yrs, concentration 1.212


[write] outputs/3_ch4_2_rq2_expertise/fig14b_concentration_by_ridge_R5_gateOFF.png
R5 gate OFF: entire process at 14.28 yrs, concentration 1.466


### Figure 15 – Ridge task expertise vs task-level observed exposure, R5 (a gate ON, b gate OFF)

In [6]:
for g, letter in [(True, "a"), (False, "b")]:
    d = task_exp[(task_exp["release"] == 5) & (task_exp["gate_on"] == g)].copy()
    d["ridge_task_years"] = d["task_id"].map(ridge)
    fig, ax = plt.subplots(figsize=(9, 7))
    rc.clean_axes(ax)
    ax.scatter(d["ridge_task_years"], d["r_tilde"], s=rc.bubble_sizes(d["WorkUsage"]), color=d["subprocess"].map(rc.SUBPROCESS_COLORS),
               alpha=0.65, edgecolor="white", linewidth=0.6, zorder=3)
    xlim = (d["ridge_task_years"].min(), d["ridge_task_years"].max())
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["ridge_task_years"], d["r_tilde"], xlim)
    rc.record_fit("Figure 15", f"R5 {gate_txt(g)}", "task ridge expertise", "r_tilde", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.TEXT_BLACK, lab.replace("\n", ", "), "line")] +
                  [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER], loc="upper left", ncol=1)
    ax.set_xlabel("Ridge Expertise (yrs, task-level)")
    ax.set_ylabel("Observed Exposure (task-level, r_tilde)")
    ax.set_title(f"Ridge expertise vs. task-level observed exposure -- Release 5, {gate_txt(g)}", loc="left", fontsize=12.5)
    fig.tight_layout()
    rc.save_figure(fig, "rq2", f"fig15{letter}_ridge_vs_task_exposure_R5_{gate_tag(g)}.png")

[write] outputs/3_ch4_2_rq2_expertise/fig15a_ridge_vs_task_exposure_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig15b_ridge_vs_task_exposure_R5_gateOFF.png


### Figure 16 – Task dispersion contribution by ridge task expertise, R5 (a gate ON, b gate OFF)

In [7]:
td = task_disp.copy()
td["ridge_task_years"] = td["task_id"].map(ridge)
td["wage_hourly_mean"] = td["task_id"].map(t["wage_hourly_mean"])


def task_dispersion_chart(x_col, x_label, fig_no, name):
    xlim, ylim = rc.global_limits([td[(td["release"] == r) & (td["gate_on"] == g)] for r, g in panels], x_col,
                                  "subprocess_weighted_dispersion", 0.08, 0.12)
    for g, letter in [(True, "a"), (False, "b")]:
        d = td[(td["release"] == 5) & (td["gate_on"] == g)].dropna(subset=[x_col])
        fig, ax = plt.subplots(figsize=(9, 7))
        rc.clean_axes(ax)
        rc.task_bubbles(ax, d, x_col, "subprocess_weighted_dispersion")
        slope, icpt, r2, n, lab = rc.trend_line(ax, d[x_col], d["subprocess_weighted_dispersion"], xlim)
        rc.record_fit(f"Figure {fig_no}", f"R5 {gate_txt(g)}", x_col, "task dispersion", slope, icpt, r2, n)
        rc.legend_box(ax, [(rc.MODE_COLOR_COOL, lab, "line")] + [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER],
                      loc="upper left", ncol=1)
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.set_xlabel(x_label)
        ax.set_ylabel("Task dispersion contribution (subprocess-level)")
        ax.set_title(f"Task dispersion by {name} -- Release 5, {gate_txt(g)}")
        rc.save_figure(fig, "rq2", f"fig{fig_no}{letter}_task_dispersion_by_{name.replace(' ', '_')}_R5_{gate_tag(g)}.png")


task_dispersion_chart("ridge_task_years", "Task ridge expertise (yrs)", 16, "ridge expertise")

[write] outputs/3_ch4_2_rq2_expertise/fig16a_task_dispersion_by_ridge_expertise_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig16b_task_dispersion_by_ridge_expertise_R5_gateOFF.png


### Expertise level of the entire process (Section 4.2.2)

Three positions on the ridge-expertise axis are written to `text_values_4_2_2.csv`:
* `process_ridge_time_weighted` — where the entire-process square sits on the x-axis of Figures 13/14 (time-weighted mean ridge expertise of all in-scope tasks);
* `ridge_weighted_by_process_dispersion` — the expertise at which process-level concentration actually sits (each task weighted by its contribution to process concentration);
* `ridge_weighted_by_exposure` — the expertise at which observed exposure sits (time × r̃ weighted).

In [8]:
vals = []
for g in rc.GATE_STATES:
    d = td[(td["release"] == 5) & (td["gate_on"] == g)]
    vals.append({"release": 5, "gate": gate_txt(g),
                 "process_ridge_time_weighted": sub_attr.loc["Entire Process", "ridge_years"],
                 "ridge_weighted_by_process_dispersion": rc._weighted_mean(d["ridge_task_years"], d["process_weighted_dispersion"]),
                 "ridge_weighted_by_subprocess_dispersion": rc._weighted_mean(d["ridge_task_years"], d["subprocess_weighted_dispersion"]),
                 "ridge_weighted_by_exposure": rc._weighted_mean(d["ridge_task_years"], d["task_time_fraction"] * d["r_tilde"]),
                 "process_concentration": float(proc_conc[(proc_conc["release"] == 5) & (proc_conc["gate_on"] == g)]["concentration"].iloc[0])})
text_vals = pd.DataFrame(vals).round(3)
rc.save_csv(text_vals, "rq2", "text_values_4_2_2.csv")
text_vals

[write] outputs/3_ch4_2_rq2_expertise/text_values_4_2_2.csv


,release,gate,process_ridge_time_weighted,ridge_weighted_by_process_dispersion,ridge_weighted_by_subprocess_dispersion,ridge_weighted_by_exposure,process_concentration
0,5,gate ON,14.284,14.115,14.334,14.125,1.212
1,5,gate OFF,14.284,14.175,14.385,14.374,1.466


### Figure 17 – Subprocess concentration by subprocess value, R5 (a gate ON, b gate OFF)
The entire-process square is drawn as in Figures 7, 13 and 14.

In [9]:
xlim17, ylim17 = rc.global_limits([sc[(sc["release"] == r) & (sc["gate_on"] == g)] for r, g in panels], "value_usd", "concentration")
ylim17 = (ylim17[0], max(ylim17[1], proc_conc["concentration"].max() * 1.15))
for g, letter in [(True, "a"), (False, "b")]:
    d = sc[(sc["release"] == 5) & (sc["gate_on"] == g)]
    pc = proc_conc[(proc_conc["release"] == 5) & (proc_conc["gate_on"] == g)].iloc[0]
    fig, ax = plt.subplots(figsize=(8, 6.5))
    rc.clean_axes(ax)
    rc.subprocess_bubbles(ax, d, "value_usd", "concentration")
    rc.draw_entire_process(ax, sub_attr.loc["Entire Process", "value_usd"], pc["concentration"])
    slope, icpt, r2, n, lab = rc.trend_line(ax, d["value_usd"], d["concentration"], xlim17)
    rc.record_fit("Figure 17", f"R5 {gate_txt(g)}", "subprocess value", "concentration", slope, icpt, r2, n)
    rc.legend_box(ax, [(rc.MODE_COLOR_COOL, lab, "line")], loc="upper left")
    ax.set_xlim(*xlim17)
    ax.set_ylim(*ylim17)
    ax.set_xlabel("Subprocess value (time-weighted hourly wage, USD)")
    ax.set_ylabel("Subprocess concentration")
    ax.set_title(f"Subprocess concentration by value -- Release 5, {gate_txt(g)}")
    rc.save_figure(fig, "rq2", f"fig17{letter}_concentration_by_value_R5_{gate_tag(g)}.png")

[write] outputs/3_ch4_2_rq2_expertise/fig17a_concentration_by_value_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig17b_concentration_by_value_R5_gateOFF.png


### Figure 18 – Task dispersion contribution by task value (occupation hourly wage), R5 (a gate ON, b gate OFF)

In [10]:
task_dispersion_chart("wage_hourly_mean", "Task value (occupation hourly wage, USD)", 18, "value")

[write] outputs/3_ch4_2_rq2_expertise/fig18a_task_dispersion_by_value_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig18b_task_dispersion_by_value_R5_gateOFF.png


## 4.2.3 Effective exposure relative to exposure patterns

### Figure 19 – Effective exposure vs task coverage, by subprocess (a = Release 4 left, b = Release 5 right) and Table 4

In [11]:
def effective_scatter(df, x_col, x_label, title, section, filename, x_scale=1.0):
    fig, ax = plt.subplots(figsize=(8.5, 8))
    ax.plot([0, 100], [0, 100], linestyle="--", color=rc.BASELINE_COLOR, linewidth=1.3, zorder=1)
    for _, row in df.iterrows():
        sp = row["subprocess"]
        is_proc = sp == "Entire Process"
        color = rc.ENTIRE_PROCESS_COLOR if is_proc else rc.SUBPROCESS_COLORS[sp]
        x, y = row[x_col] * x_scale, row["effective_exposure_pct"]
        ax.scatter(x, y, color=color, marker="s" if is_proc else "o", s=160 if is_proc else 130, zorder=3,
                   edgecolor="white", linewidth=1.0)
        ax.annotate(sp, (x, y), xytext=(8, 6), textcoords="offset points", fontsize=9.5, color=color,
                    fontweight="semibold", path_effects=rc.TEXT_HALO, zorder=4)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel(x_label)
    ax.set_ylabel("Effective exposure (%)")
    rc.clean_axes(ax)
    ax.set_title(title, loc="left", fontsize=12.5)
    fig.tight_layout()
    rc.save_figure(fig, section, filename)


for r, letter in [(4, "a"), (5, "b")]:
    effective_scatter(eff[eff["release"] == r], "task_coverage_pct", "Task coverage (% of in-scope tasks touched, either platform)",
                      f"Effective exposure by task coverage, by subprocess -- Release {r} (time-fraction weighted)",
                      "rq2", f"fig19{letter}_effective_vs_coverage_R{r}.png")

scope_order = ["Entire Process"] + rc.SUBPROCESS_ORDER
eff["eff_minus_cov"] = eff["effective_exposure_pct"] - eff["task_coverage_pct"]
table4 = eff.pivot(index="subprocess", columns="release", values="eff_minus_cov").reindex(scope_order)
table4.columns = [f"Release {c}" for c in table4.columns]
rc.save_csv(table4.round(1).reset_index().rename(columns={"subprocess": "Scope"}), "rq2", "table04_effective_minus_coverage.csv")
eff.pivot(index="subprocess", columns="release", values=["task_coverage_pct", "effective_exposure_pct", "eff_minus_cov"]).reindex(scope_order).round(1)

findfont: Failed to find font weight semibold for Arial, now using 700.


[write] outputs/3_ch4_2_rq2_expertise/fig19a_effective_vs_coverage_R4.png


[write] outputs/3_ch4_2_rq2_expertise/fig19b_effective_vs_coverage_R5.png
[write] outputs/3_ch4_2_rq2_expertise/table04_effective_minus_coverage.csv


task_coverage_pct       effective_exposure_pct       eff_minus_cov      
release                                     4     5                      4     5             4     5
subprocess                                                                                          
Entire Process                           42.9  45.1                   23.4  25.7         -19.5 -19.3
Master Data Management                   62.5  75.0                   21.9  37.7         -40.6 -37.3
Record                                   48.4  41.9                   25.7  26.5         -22.7 -15.4
Close & Consolidation                    40.0  40.0                   31.5  29.2          -8.5 -10.8
Report                                   15.4  30.8                   10.5   9.3          -4.9 -21.5
Analysis & Business Insight              58.3  58.3                   30.4  35.9         -27.9 -22.4
Planning & Budgeting                     36.4  40.9                   24.0  26.9         -12.3 -14.0

### Figure 20 – Effective exposure vs observed exposure (Rₒ × 100), R5 (a gate ON, b gate OFF) and Table 5

In [12]:
obs = pd.concat([sub_exp[["subprocess", "release", "gate_on", "R_o"]],
                 proc_exp.assign(subprocess="Entire Process").rename(columns={"R_process": "R_o"})[["subprocess", "release", "gate_on", "R_o"]]])
eo = eff.merge(obs, on=["subprocess", "release"])
eo["eff_minus_obs"] = eo["effective_exposure_pct"] - 100 * eo["R_o"]
for g, letter in [(True, "a"), (False, "b")]:
    effective_scatter(eo[(eo["release"] == 5) & (eo["gate_on"] == g)], "R_o",
                      f"Observed exposure, R_o (%) (gate={'ON' if g else 'OFF'})",
                      f"Effective exposure by observed exposure, by subprocess -- Release 5 (gate={'ON' if g else 'OFF'})",
                      "rq2", f"fig20{letter}_effective_vs_observed_R5_{gate_tag(g)}.png", x_scale=100)
table5 = eo.pivot_table(index="subprocess", columns=["gate_on", "release"], values="eff_minus_obs").reindex(scope_order)
table5 = table5[[(True, 4), (True, 5), (False, 4), (False, 5)]]
table5.columns = [f"Gate {'ON' if g else 'OFF'} R{r}" for g, r in table5.columns]
rc.save_csv(table5.round(1).reset_index().rename(columns={"subprocess": "Scope"}), "rq2", "table05_effective_minus_observed.csv")
table5.round(1)

[write] outputs/3_ch4_2_rq2_expertise/fig20a_effective_vs_observed_R5_gateON.png


[write] outputs/3_ch4_2_rq2_expertise/fig20b_effective_vs_observed_R5_gateOFF.png
[write] outputs/3_ch4_2_rq2_expertise/table05_effective_minus_observed.csv


,Gate ON R4,Gate ON R5,Gate OFF R4,Gate OFF R5
subprocess,,,,
Entire Process,7.2,6.0,-9.2,-9.4
Master Data Management,-3.6,12.4,-28.6,-18.0
Record,4.9,-5.4,-14.0,-18.0
Close & Consolidation,-12.8,-11.1,-12.8,-11.1
Report,4.4,3.5,-5.0,-11.9
Analysis & Business Insight,12.1,17.8,-16.1,-15.4
Planning & Budgeting,15.4,18.2,4.0,10.6


### Table D7 and Figure D5 – Task-level decomposition of the coverage gap

`gap_pp` = effective exposure − task coverage (Table 4) = `weight_component_pp` (touched tasks carry less (−) or more (+) time than their count share — *low-weight* tasks) + `success_component_pp` (shortfall from touched tasks that Claude does not reliably complete — *low-success* tasks). Counts of touched tasks that are low-weight, low-success, or both are included.

In [13]:
decomp = rc.decompose_coverage_gap(tm, panel)
decomp["subprocess"] = pd.Categorical(decomp["subprocess"], scope_order, ordered=True)
decomp = decomp.sort_values(["release", "subprocess"])
rc.save_csv(decomp.round(2), "appD", "tableD7_coverage_gap_decomposition.csv")

# Figure D5: each task's raw usage share vs its effective contribution (weight x success), R5
r = 5
usage = rc.task_series(panel, r, rc.RELEASE_PLATFORMS[r], "usage", "count", agg="sum").reindex(t.index).fillna(0)
succ = rc.task_success_table(panel, r)["success_pooled"].reindex(t.index).fillna(0)
w = t["task_time_fraction"] / t["task_time_fraction"].sum()
contrib = pd.DataFrame({"subprocess": t["subprocess"], "usage_share_pct": 100 * usage / usage.sum(),
                        "success_pct": succ, "weight": w, "effective_contribution": w * succ})
rc.save_csv(contrib.reset_index(), "appD", "figD5_task_contribution_R5.csv")
fig, ax = plt.subplots(figsize=(9, 7.5))
for sp in rc.SUBPROCESS_ORDER:
    s = contrib[contrib["subprocess"] == sp]
    ax.scatter(s["usage_share_pct"], s["effective_contribution"], color=rc.SUBPROCESS_COLORS[sp], s=32, alpha=0.75,
               edgecolor="white", linewidth=0.4, zorder=3)
ax.axvline(contrib["usage_share_pct"].median(), color=rc.LIGHT_GREY, linewidth=1.1, linestyle="--", zorder=1.5)
ax.axhline(contrib["effective_contribution"].median(), color=rc.LIGHT_GREY, linewidth=1.1, linestyle="--", zorder=1.5)
ax.set_xlim(0, contrib["usage_share_pct"].max() * 1.05)
ax.set_ylim(0, contrib["effective_contribution"].max() * 1.05)
ax.set_xlabel("Raw usage share (%, both platforms combined)")
ax.set_ylabel("Effective contribution (weight × success, pp)")
rc.clean_axes(ax)
ax.set_title("Task-level usage vs. effective contribution -- Release 5 (time-fraction weighted)", loc="left", fontsize=12.5)
rc.legend_box(ax, [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER], loc="upper right", ncol=1, fontsize=8.5)
fig.tight_layout()
rc.save_figure(fig, "appD", "figD5_task_contribution_R5.png")
decomp[["release", "subprocess", "gap_pp", "weight_component_pp", "success_component_pp", "residual_pp",
        "touched_low_weight_n", "touched_low_success_n", "touched_low_weight_and_low_success_n"]].round(1)

[write] outputs/4_appendix_D/tableD7_coverage_gap_decomposition.csv
[write] outputs/4_appendix_D/figD5_task_contribution_R5.csv


[write] outputs/4_appendix_D/figD5_task_contribution_R5.png


,release,subprocess,gap_pp,weight_component_pp,success_component_pp,residual_pp,touched_low_weight_n,touched_low_success_n,touched_low_weight_and_low_success_n
0,4,Entire Process,-19.5,1.2,-20.7,0.0,18,13,5
1,4,Master Data Management,-40.6,-8.3,-32.3,0.0,3,2,1
2,4,Record,-22.7,4.5,-27.2,0.0,7,7,4
3,4,Close & Consolidation,-8.5,6.0,-14.6,0.0,0,0,0
4,4,Report,-4.9,1.0,-5.8,0.0,1,0,0
5,4,Analysis & Business Insight,-27.9,1.6,-29.5,0.0,3,3,0
6,4,Planning & Budgeting,-12.3,0.3,-12.6,0.0,4,1,0
7,5,Entire Process,-19.3,1.6,-21.0,0.0,18,9,4
8,5,Master Data Management,-37.3,-1.7,-35.6,0.0,4,1,1
9,5,Record,-15.4,5.0,-20.4,0.0,5,4,2


## 4.2.4 Determinants of success and effective exposure

### Figure 21 – Task success rate by ridge expertise, by platform (a = Release 4, b = Release 5)
Markers are binned means (15 quantile bins); dashed lines and the stats box are OLS fits on the raw task points.

In [14]:
def platform_binned_chart(release, get_xy, x_label, y_label, title, section, filename, fig_id,
                          y_cap=None, y_range=None, x_range=None, hline50=False, hline_text=False):
    fig, ax = plt.subplots(figsize=(8.5, 7))
    stats = {}
    for p in rc.PLATFORMS:
        if p not in rc.RELEASE_PLATFORMS[release]:
            continue
        x, y = get_xy(p)
        r2, n, slope, icpt = rc.binned_scatter_with_trend(ax, x.values, y.values, rc.PLATFORM_COLORS[p], rc.PLATFORM_MARKERS[p],
                                                          label=rc.PLATFORM_LABELS[p])
        stats[p] = dict(r2=r2, n=n, slope=slope, intercept=icpt)
        rc.record_fit(fig_id, f"R{release} {rc.PLATFORM_LABELS[p]}", x_label, y_label, slope, icpt, r2, n)
    if y_cap:
        rc.capped_ylim(ax, y_cap)
    if y_range:
        ax.set_ylim(*y_range)
    if x_range:
        ax.set_xlim(*x_range)
    if hline50:
        ax.axhline(50, color=rc.BASELINE_COLOR, linewidth=rc.BASELINE_LINEWIDTH, linestyle="--", zorder=1.5)
        if hline_text:
            ax.text(ax.get_xlim()[1], 50, " 50% success ", ha="right", va="bottom", fontsize=9, color=rc.BASELINE_COLOR, fontstyle="italic")
    rc.platform_stats_box(ax, stats)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    rc.clean_axes(ax)
    ax.set_title(title, loc="left", fontsize=13)
    fig.tight_layout()
    rc.save_figure(fig, section, filename)
    return stats


def success(release, p):
    return rc.get_task_metric(panel, release, p, "task_success", "yes", "pct") * 100


for r, letter in [(4, "a"), (5, "b")]:
    platform_binned_chart(r, lambda p, r=r: (ridge.reindex(success(r, p).index), success(r, p)),
                          "Ridge Expertise (yrs)", "Task Success Rate (%)",
                          f"Success rates by ridge expertise, by platform -- Release {r}", "rq2",
                          f"fig21{letter}_success_by_ridge_R{r}.png", "Figure 21", y_cap=100, hline50=True)

[write] outputs/3_ch4_2_rq2_expertise/fig21a_success_by_ridge_R4.png


[write] outputs/3_ch4_2_rq2_expertise/fig21b_success_by_ridge_R5.png


### Table 6 – Average task primitives, entire R2R process, by platform, R4 vs R5

Use-case shares are **conversation-pooled**: the conversation counts of every in-scope task are summed per category (work, personal, coursework, not classified) and divided by the summed total, so the four rows sum to exactly 100%. All other rows are unweighted means across tasks with a reading (as in the caption).

`table06_use_case_diagnostic.csv` sets the pooled shares beside the unweighted mean of each category's pct over the tasks that report it. Those means do not sum to 100%, because each category's mean is taken over a different set of tasks.

In [15]:
PRIMS = [  # label, dimension, category, variable, scale
    ("Human education (years)", "human_education_years", None, "mean", 1),
    ("AI education (years)", "ai_education_years", None, "mean", 1),
    ("AI autonomy (1 to 5)", "ai_autonomy", None, "mean", 1),
    ("Human-only time (minutes)", "human_only_time", None, "mean", 60),
    ("Human-with-AI time (minutes)", "human_with_ai_time", None, "mean", 1),
    ("Human cannot complete alone (%)", "human_only_ability", "no", "pct", 100),
    ("Task success rate (%)", "task_success", "yes", "pct", 100),
]
USE_CASES = [("work", "Work (%)"), ("personal", "Personal (%)"), ("coursework", "Coursework (%)"), ("not_classified", "Not classified (%)")]


def use_case_pooled(release, p, ids=None):
    """Conversation-pooled use-case shares (%) over the given tasks; sums to 100."""
    ids = set(t.index if ids is None else ids)
    m = (panel["release"] == release) & (panel["dimension_n"] == "use_case") & (panel["platform_n"] == p) & panel["task_id"].isin(ids)
    cnt = panel.loc[m & (panel["variable_n"] == "count")].groupby("category_n")["value"].sum()
    if cnt.sum() <= 0:   # fallback: pct x task usage count
        pct = panel.loc[m & (panel["variable_n"] == "pct")]
        n = rc.task_series(panel, release, p, "usage", "count", agg="sum")
        cnt = (pct.assign(n=pct["task_id"].map(n)).eval("value * n")).groupby(pct["category_n"]).sum()
    return 100 * cnt / cnt.sum()


def primitives_table(scope=None):
    ids = rc.scope_task_ids(tm, scope)
    rows = []
    for p in rc.PLATFORMS:
        for label, dim, cat, var, scale in PRIMS:
            v = {r: rc.get_task_metric(panel, r, p, dim, cat, var).reindex(ids).mean() * scale for r in (4, 5)}
            rows.append({"platform": rc.PLATFORM_LABELS[p], "primitive": label, "R4": v[4], "R5": v[5], "delta": v[5] - v[4]})
        uc = {r: use_case_pooled(r, p, ids) for r in (4, 5)}
        for cat, label in USE_CASES:
            v4, v5 = uc[4].get(cat, 0.0), uc[5].get(cat, 0.0)
            rows.append({"platform": rc.PLATFORM_LABELS[p], "primitive": label, "R4": v4, "R5": v5, "delta": v5 - v4})
    return pd.DataFrame(rows)


table6 = primitives_table()
rc.save_csv(table6.round(2), "rq2", "table06_primitives.csv")

diag = []
for p in rc.PLATFORMS:
    for r in (4, 5):
        pooled = use_case_pooled(r, p)
        for cat, label in USE_CASES:
            task_mean = rc.get_task_metric(panel, r, p, "use_case", cat, "pct").reindex(t.index)
            diag.append({"platform": rc.PLATFORM_LABELS[p], "release": r, "category": cat,
                         "unweighted_task_mean_pct": 100 * task_mean.mean(), "n_tasks_with_row": int(task_mean.notna().sum()),
                         "conversation_pooled_pct": pooled.get(cat, 0.0)})
diag = pd.DataFrame(diag)
rc.save_csv(diag.round(2), "rq2", "table06_use_case_diagnostic.csv")
print(diag.groupby(["platform", "release"])[["unweighted_task_mean_pct", "conversation_pooled_pct"]].sum().round(1))
table6.pivot_table(index="primitive", columns="platform", values=["R4", "R5", "delta"], sort=False).round(2)

[write] outputs/3_ch4_2_rq2_expertise/table06_primitives.csv
[write] outputs/3_ch4_2_rq2_expertise/table06_use_case_diagnostic.csv
                   unweighted_task_mean_pct  conversation_pooled_pct
platform  release                                                   
1P API    4                           127.5                    100.0
          5                           127.8                    100.0
Claude.ai 4                           156.4                    100.0
          5                           137.2                    100.0


R4               R5            delta       
platform                        Claude.ai 1P API Claude.ai 1P API Claude.ai 1P API
primitive                                                                         
Human education (years)             13.17  13.56     13.34  13.77      0.17   0.21
AI education (years)                12.94  12.79     13.25  13.07      0.31   0.28
AI autonomy (1 to 5)                 3.13   2.79      3.23   2.72      0.10  -0.07
Human-only time (minutes)          187.52  79.35    214.27  94.22     26.74  14.87
Human-with-AI time (minutes)        15.67   3.22     15.52   3.35     -0.15   0.13
Human cannot complete alone (%)      9.73   7.66     17.34   7.48      7.61  -0.19
Task success rate (%)               67.82  58.81     68.42  54.29      0.60  -4.52
Work (%)                            62.61  87.35     71.81  88.77      9.20   1.42
Personal (%)                        15.03   6.91     18.24   5.88      3.22  -1.04
Coursework (%)                      17.12   2.52      6.82   2.13    -10.30  -0.39
Not classified (%)                   5.25   3.22      3.13   3.23     -2.12   0.01

### Table D2 – Task success rate by subprocess, platform and release
Unweighted mean across tasks with a success reading (same convention as Table 6; the Close & Consolidation 1P API change is the −17.71 pp cited in the text).

In [16]:
rows = []
for scope in scope_order:
    ids = rc.scope_task_ids(tm, scope)
    for p in rc.PLATFORMS:
        s4, s5 = success(4, p).reindex(ids), success(5, p).reindex(ids)
        rows.append({"subprocess": scope, "platform": rc.PLATFORM_LABELS[p], "n_R4": int(s4.notna().sum()), "R4": s4.mean(),
                     "n_R5": int(s5.notna().sum()), "R5": s5.mean(), "delta_pp": s5.mean() - s4.mean()})
tableD2 = pd.DataFrame(rows)
rc.save_csv(tableD2.round(2), "appD", "tableD2_success_by_subprocess.csv")
tableD2.round(2)

[write] outputs/4_appendix_D/tableD2_success_by_subprocess.csv


,subprocess,platform,n_R4,R4,n_R5,R5,delta_pp
0,Entire Process,Claude.ai,27,67.82,31,68.42,0.60
1,Entire Process,1P API,19,58.81,20,54.29,-4.52
2,Master Data Management,Claude.ai,3,74.38,4,69.55,-4.83
3,Master Data Management,1P API,4,53.45,5,59.79,6.34
4,Record,Claude.ai,8,69.65,10,67.12,-2.53
5,Record,1P API,6,60.92,8,50.95,-9.98
6,Close & Consolidation,Claude.ai,2,57.14,2,64.79,7.65
7,Close & Consolidation,1P API,2,69.98,2,52.27,-17.71
8,Report,Claude.ai,2,59.16,2,58.39,-0.77
9,Report,1P API,2,64.93,1,46.64,-18.29


### Figure 22 – Human-only time by ridge expertise (a) and task success by human-only time (b), R5

In [17]:
def hot(release, p):
    return rc.get_task_metric(panel, release, p, "human_only_time")


platform_binned_chart(5, lambda p: (ridge.reindex(hot(5, p).index), hot(5, p)), "Ridge Expertise (yrs)", "Human-only Time (hours)",
                      "Human-only time by ridge expertise, by platform -- Release 5", "rq2",
                      "fig22a_human_only_time_by_ridge_R5.png", "Figure 22")
platform_binned_chart(5, lambda p: (hot(5, p), success(5, p).reindex(hot(5, p).index)), "Human-only Time (hours)", "Task Success Rate (%)",
                      "Task success by human-only time, by platform -- Release 5", "rq2",
                      "fig22b_success_by_human_only_time_R5.png", "Figure 22", y_cap=100, hline50=True, hline_text=True)

[write] outputs/3_ch4_2_rq2_expertise/fig22a_human_only_time_by_ridge_R5.png


[write] outputs/3_ch4_2_rq2_expertise/fig22b_success_by_human_only_time_R5.png


{'claude_ai': {'r2': 0.2931012834016443,
  'n': 31,
  'slope': -3.9548708194924296,
  'intercept': 82.35004376089408},
 '1p_api': {'r2': 0.0748114008063524,
  'n': 20,
  'slope': -3.6401781529099067,
  'intercept': 59.004112489555666}}

### Figure 23 – AI autonomy by ridge expertise (a) and task success by AI autonomy (b), R5

In [18]:
def auto(release, p):
    return rc.get_task_metric(panel, release, p, "ai_autonomy")


platform_binned_chart(5, lambda p: (ridge.reindex(auto(5, p).index), auto(5, p)), "Ridge Expertise (yrs)", "AI Autonomy (1-5)",
                      "Autonomy by ridge expertise, by platform -- Release 5", "rq2",
                      "fig23a_autonomy_by_ridge_R5.png", "Figure 23", y_range=(1, 5))
platform_binned_chart(5, lambda p: (auto(5, p), success(5, p).reindex(auto(5, p).index)), "AI Autonomy (1-5)", "Task Success Rate (%)",
                      "Task success by autonomy, by platform -- Release 5", "rq2",
                      "fig23b_success_by_autonomy_R5.png", "Figure 23", y_cap=100, x_range=(1, 5), hline50=True, hline_text=True)

[write] outputs/3_ch4_2_rq2_expertise/fig23a_autonomy_by_ridge_R5.png


[write] outputs/3_ch4_2_rq2_expertise/fig23b_success_by_autonomy_R5.png


{'claude_ai': {'r2': 0.05039644403909194,
  'n': 31,
  'slope': -9.045272983906285,
  'intercept': 97.68238767324995},
 '1p_api': {'r2': 0.29899929971379413,
  'n': 20,
  'slope': -24.144170963236338,
  'intercept': 119.4031507651092}}

### Table 7 – Stepwise OLS of Claude.ai task success (task × release, R4–R5 pooled)
Standard errors: **conventional (homoskedastic) OLS**; **no release indicator**. Robustness with task-clustered SEs and a release indicator is in Table D8a; per-release estimates in Table D10.

In [19]:
T7_ROWS = [("expertise", "Ridge expertise (years)"), ("autonomy", "AI autonomy (1 to 5)"),
           ("human_only_time", "Human-only time (hours)"), ("release_5", "Release 5 indicator"), ("intercept", "Constant")]
T8_ROWS = [("ln_input_index", "ln(input context-length index)"), ("ln_output_index", "ln(output context-length index)"),
           ("autonomy", "AI autonomy (1 to 5)"), ("release_5", "Release 5 indicator"), ("intercept", "Constant")]


def stepwise(df, base_specs, rows, extra=(), cluster=None, ftest=True):
    df, dcols = rc.add_subprocess_dummies(df)
    df["release_5"] = (df["release"] == 5).astype(float)
    specs = [(lab, cols + list(extra)) for lab, cols in base_specs]
    specs.append(("(4) + Subprocess FE", base_specs[-1][1] + list(extra) + dcols))
    full = specs[-1][1]
    res = rc.run_stepwise(df, "task_success_pct", specs, cluster_col=cluster,
                          fe_test=(base_specs[-1][1] + list(extra), full) if ftest else None)
    return rc.regression_table(res, [r_ for r_ in rows if r_[0] in {c for _, cs in specs for c in cs} | {"intercept"}], fe_from=3)


T7_SPECS = [("(1) Expertise only", ["expertise"]), ("(2) + Autonomy", ["expertise", "autonomy"]),
            ("(3) + Human-only time", ["expertise", "autonomy", "human_only_time"])]
claude_df = rc.build_claude_success_dataset(tm, panel)
table7 = stepwise(claude_df, T7_SPECS, T7_ROWS)
rc.save_csv(table7, "rq2", "table07_claude_success_regression.csv")
table7

[write] outputs/3_ch4_2_rq2_expertise/table07_claude_success_regression.csv


,,(1) Expertise only,(2) + Autonomy,(3) + Human-only time,(4) + Subprocess FE
0,Ridge expertise (years),-0.217,0.428,2.082*,-0.256
1,,(1.132),(1.273),(1.095),(1.409)
2,AI autonomy (1 to 5),,-6.943,10.530*,6.289
3,,,(6.316),(6.188),(6.453)
4,Human-only time (hours),,,-5.741***,-5.815***
5,,,,(1.102),(1.263)
6,Constant,71.293***,84.087***,23.876,70.105**
7,,(16.540),(20.200),(20.252),(28.091)
8,Subprocess fixed effects,No,No,No,Yes
9,F-test of fixed effects,,,,"F(5, 49) = 1.81 [p = 0.128]"


### Figures 24, 25, D4 and D2 – Context length on the 1P API
Context-length indices are the AEI prompt-token (input) and completion-token (output) indices re-indexed to the R2R task set and log-transformed (0 = R2R average). Axis ranges are shared across releases.

In [20]:
def api_frame(release):
    d = r2r_idx[r2r_idx["release"] == release].set_index("task_id")
    d = d[(d["input_index"] > 0) & (d["output_index"] > 0)].copy()
    d["ln_input"], d["ln_output"] = np.log(d["input_index"]), np.log(d["output_index"])
    d["success"] = success(release, "1p_api").reindex(d.index)
    d["expertise"] = ridge.reindex(d.index)
    return d


def ln_range(col):
    v = pd.concat([api_frame(r)[col] for r in rc.EXPOSURE_RELEASES])
    pad = (v.max() - v.min()) * 0.05
    return (v.min() - pad, v.max() + pad)


IN_RANGE, OUT_RANGE = ln_range("ln_input"), ln_range("ln_output")


def context_chart(release, y_col, y_label, title, section, filename, fig_id, y_cap=None):
    df = api_frame(release).dropna(subset=[y_col])
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 7), sharey=True)
    for (x_col, x_label, xr), ax in zip([("ln_input", "ln(Input Context Length Index, R2R-relative)", IN_RANGE),
                                         ("ln_output", "ln(Output Context Length Index, R2R-relative)", OUT_RANGE)], axes):
        rc.scatter_by_subprocess(ax, df, x_col, y_col)
        slope, icpt, r2, n = rc.fit_line(df[x_col], df[y_col])
        rc.record_fit(fig_id, f"R{release} {x_col}", x_col, y_col, slope, icpt, r2, n)
        xl = np.linspace(df[x_col].min(), df[x_col].max(), 60)
        ax.plot(xl, slope * xl + icpt, color=rc.BASELINE_COLOR, linestyle="--", linewidth=1.6, zorder=2)
        unit = "pp" if y_col == "success" else "yrs"
        ax.text(0.03, 0.03 if y_col == "success" else 0.97, f"R² = {r2:.3f}  |  slope = {slope:.2f} {unit} / ln-unit  |  n = {n}",
                transform=ax.transAxes, va="bottom" if y_col == "success" else "top", ha="left", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=rc.LEGEND_EDGE_COLOR, linewidth=0.8))
        ax.set_xlabel(x_label)
        ax.set_xlim(*xr)
        if y_cap:
            rc.capped_ylim(ax, y_cap)
        rc.clean_axes(ax)
    axes[0].set_ylabel(y_label)
    rc.legend_box(fig, [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER if (df["subprocess"] == sp).any()],
                  loc="lower center", bbox_to_anchor=(0.5, -0.04), ncol=3)
    fig.suptitle(title, fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0.06, 1, 0.95])
    rc.save_figure(fig, section, filename)


context_chart(5, "success", "Task Success Rate (%)", "Context length by success rate -- Release 5", "rq2",
              "fig24_context_vs_success_R5.png", "Figure 24", y_cap=100)
context_chart(5, "expertise", "Ridge Expertise (yrs)", "Context length by ridge expertise -- Release 5", "rq2",
              "fig25_context_vs_ridge_R5.png", "Figure 25")
context_chart(4, "success", "Task Success Rate (%)", "Context length by success rate -- Release 4", "appD",
              "figD4_context_vs_success_R4.png", "Figure D4", y_cap=100)

# Figure D2 -- ln(input index) vs ln(output index), R5
d = api_frame(5)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
rc.scatter_by_subprocess(ax, d, "ln_input", "ln_output")
slope, icpt, r2, n = rc.fit_line(d["ln_input"], d["ln_output"])
rc.record_fit("Figure D2", "R5", "ln_input", "ln_output", slope, icpt, r2, n)
xl = np.linspace(d["ln_input"].min(), d["ln_input"].max(), 60)
ax.plot(xl, slope * xl + icpt, color=rc.BASELINE_COLOR, linestyle="--", linewidth=rc.BASELINE_LINEWIDTH, zorder=2)
rc.legend_box(ax, [(rc.SUBPROCESS_COLORS[sp], f"{sp} (N={(d['subprocess'] == sp).sum()})", "o") for sp in rc.SUBPROCESS_ORDER
                   if (d["subprocess"] == sp).any()] + [(rc.BASELINE_COLOR, f"Best fit (R² = {r2:.3f}, β = {slope:.3f})", "line")],
              loc="upper left", ncol=1)
ax.set_xlabel("ln(Input Token Index, R2R-relative)")
ax.set_ylabel("ln(Output Token Index, R2R-relative)")
ax.set_xlim(*IN_RANGE)
ax.set_ylim(*OUT_RANGE)
rc.clean_axes(ax)
ax.set_title("Output token index vs. input token index across R2R tasks -- Release 5", loc="left", fontsize=13)
fig.tight_layout()
rc.save_figure(fig, "appD", "figD2_input_vs_output_index_R5.png")

[write] outputs/3_ch4_2_rq2_expertise/fig24_context_vs_success_R5.png


[write] outputs/3_ch4_2_rq2_expertise/fig25_context_vs_ridge_R5.png


[write] outputs/4_appendix_D/figD4_context_vs_success_R4.png


[write] outputs/4_appendix_D/figD2_input_vs_output_index_R5.png


### Table 8 (pooled), Table D1 (by release) – OLS of 1P API task success
Standard errors: conventional OLS; no release indicator. The output-index coefficient reaches p < 0.10 only in specification (4).

In [21]:
T8_SPECS = [("(1) Input index", ["ln_input_index"]), ("(2) + Output index", ["ln_input_index", "ln_output_index"]),
            ("(3) + Autonomy", ["ln_input_index", "ln_output_index", "autonomy"])]
api_df = rc.build_api_success_dataset(tm, panel)
table8 = stepwise(api_df, T8_SPECS, T8_ROWS)
rc.save_csv(table8, "rq2", "table08_api_success_regression.csv")

d1 = []
for r in rc.EXPOSURE_RELEASES:
    tab = stepwise(api_df[api_df["release"] == r], T8_SPECS, T8_ROWS)
    tab.insert(0, "release", f"Release {r}")
    d1.append(tab)
tableD1 = pd.concat(d1, ignore_index=True)
rc.save_csv(tableD1, "appD", "tableD1_api_success_regression_by_release.csv")

d10 = []
for r in rc.EXPOSURE_RELEASES:
    tab = stepwise(claude_df[claude_df["release"] == r], T7_SPECS, T7_ROWS)
    tab.insert(0, "release", f"Release {r}")
    d10.append(tab)
rc.save_csv(pd.concat(d10, ignore_index=True), "appD", "tableD10_claude_success_regression_by_release.csv")
print(table8.to_string(index=False))
tableD1

[write] outputs/3_ch4_2_rq2_expertise/table08_api_success_regression.csv
[write] outputs/4_appendix_D/tableD1_api_success_regression_by_release.csv
[write] outputs/4_appendix_D/tableD10_claude_success_regression_by_release.csv
                                (1) Input index (2) + Output index (3) + Autonomy         (4) + Subprocess FE
 ln(input context-length index)      -14.198***         -15.471***     -12.261***                  -16.406***
                                        (3.239)            (3.563)        (3.890)                     (3.993)
ln(output context-length index)                              3.964          5.601                      9.367*
                                                           (4.552)        (4.509)                     (5.497)
           AI autonomy (1 to 5)                                          -13.767*                   -16.939**
                                                                          (7.648)                     (8.043)
   

,release,,(1) Input index,(2) + Output index,(3) + Autonomy,(4) + Subprocess FE
0,Release 4,ln(input context-length index),-10.532**,-11.869**,-8.171,-11.559*
1,Release 4,,(4.681),(4.742),(5.157),(5.720)
2,Release 4,ln(output context-length index),,9.857,6.159,5.728
3,Release 4,,,(8.054),(8.105),(15.610)
4,Release 4,AI autonomy (1 to 5),,,-19.274,-22.868
5,Release 4,,,,(12.599),(17.464)
6,Release 4,Constant,58.222***,59.091***,112.840***,122.434**
7,Release 4,,(2.823),(2.872),(35.243),(46.416)
8,Release 4,Subprocess fixed effects,No,No,No,Yes
9,Release 4,F-test of fixed effects,,,,"F(5, 10) = 1.24 [p = 0.359]"


### Table D8 – Robustness of Tables 7 and 8: task-clustered standard errors, with and without a release indicator
Each task appears up to twice in the pooled sample (R4 and R5), so conventional SEs may be too small. CR1 cluster-robust SEs by task (p-values on G − 1 df) are reported (i) for the baseline specifications and (ii) adding a Release 5 indicator.

In [22]:
for df_, specs, rows, name in [(claude_df, T7_SPECS, T7_ROWS, "tableD8a_claude_success_robustness.csv"),
                               (api_df, T8_SPECS, T8_ROWS, "tableD8b_api_success_robustness.csv")]:
    a = stepwise(df_, specs, rows, cluster="task_id", ftest=False)
    a.insert(0, "variant", "Clustered SE (task)")
    b = stepwise(df_, specs, rows, extra=["release_5"], cluster="task_id", ftest=False)
    b.insert(0, "variant", "Clustered SE (task) + release indicator")
    out = pd.concat([a, b], ignore_index=True)
    rc.save_csv(out, "appD", name)
    print(name)
    print(out.to_string(index=False))

[write] outputs/4_appendix_D/tableD8a_claude_success_robustness.csv


tableD8a_claude_success_robustness.csv
                                variant                          (1) Expertise only (2) + Autonomy (3) + Human-only time (4) + Subprocess FE
                    Clustered SE (task)  Ridge expertise (years)             -0.217          0.428                 2.082              -0.256
                    Clustered SE (task)                                     (1.257)        (1.624)               (1.234)             (1.799)
                    Clustered SE (task)     AI autonomy (1 to 5)                            -6.943                10.530               6.289
                    Clustered SE (task)                                                    (7.907)               (7.661)             (7.870)
                    Clustered SE (task)  Human-only time (hours)                                               -5.741***           -5.815***
                    Clustered SE (task)                                                                          (

### Figure 26 – Task-level usage share vs R2R-relative API cost index, R5 (a raw, b partial)
Partial regression (Frisch–Waugh–Lovell) controls: subprocess fixed effects, Claude.ai automation and augmentation shares, and task complexity.

Task complexity is ridge expertise (`COMPLEXITY_CONTROL = "ridge"`), consistent with Section 3.1.2. A sensitivity version controlling for the platform-averaged AEI `human_education_years` instead is written as `fig26b_alt_aei_education_control_R5.png`.

In [23]:
COMPLEXITY_CONTROL = "ridge"   # or "aei_education" for the sensitivity version


def cost_share_frame(release):
    d = r2r_idx[r2r_idx["release"] == release].set_index("task_id")[["cost_index", "subprocess"]]
    d["share"] = rc.get_r2r_task_pct(panel, release, "1p_api").reindex(d.index)
    d = d[(d["cost_index"] > 0) & (d["share"] > 0)].copy()
    d["ln_cost"], d["ln_share"] = np.log(d["cost_index"]), np.log(d["share"])
    return d


def partial_sample(release, control):
    d = cost_share_frame(release).join(rc.get_collaboration_shares(panel, release, "claude_ai"), how="inner")
    if control == "ridge":
        d["complexity"] = ridge.reindex(d.index)
    else:
        edu = pd.concat([rc.get_task_metric(panel, release, p, "human_education_years") for p in rc.PLATFORMS], axis=1).mean(axis=1)
        d["complexity"] = edu.reindex(d.index)
    d = d.dropna(subset=["automation_share", "augmentation_share", "complexity", "subprocess"])
    present = [sp for sp in rc.SUBPROCESS_ORDER if (d["subprocess"] == sp).any()]
    ctrl = pd.DataFrame({f"sp_{sp}": (d["subprocess"] == sp).astype(float) for sp in present[1:]}, index=d.index)
    ctrl = ctrl.join(d[["automation_share", "augmentation_share", "complexity"]])
    Xc = np.column_stack([np.ones(len(d))] + [ctrl[c].values for c in ctrl.columns])
    resid = lambda y: y - Xc @ np.linalg.lstsq(Xc, y, rcond=None)[0]
    d["x_resid"], d["y_resid"] = resid(d["ln_cost"].values), resid(d["ln_share"].values)
    return d


def cost_chart(d, x, y, x_label, y_label, title, filename, fig_id, panel_label, through_origin=False):
    slope, icpt, r2, n = rc.fit_line(d[x], d[y])
    rc.record_fit(fig_id, panel_label, x, y, slope, icpt, r2, n)
    fig, ax = plt.subplots(figsize=(9.5, 7.5))
    rc.scatter_by_subprocess(ax, d, x, y)
    xl = np.linspace(d[x].min(), d[x].max(), 60)
    ax.plot(xl, slope * xl + (0 if through_origin else icpt), color=rc.BASELINE_COLOR, linestyle="--", linewidth=rc.BASELINE_LINEWIDTH, zorder=2)
    kind = "Partial fit" if through_origin else "Best fit"
    rc.legend_box(ax, [(rc.SUBPROCESS_COLORS[sp], sp, "o") for sp in rc.SUBPROCESS_ORDER if (d["subprocess"] == sp).any()]
                  + [(rc.BASELINE_COLOR, f"{kind} (R² = {r2:.3f}, β = {slope:.3f}, n = {n})", "line")], loc="upper right", ncol=1)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    rc.clean_axes(ax)
    ax.set_title(title, loc="left", fontsize=12.5)
    fig.tight_layout()
    rc.save_figure(fig, "rq2", filename)


cost_chart(cost_share_frame(5), "ln_cost", "ln_share", "ln(R2R Cost Index)", "ln(R2R Conversation Share)",
           "Task usage share vs. API cost index (R2R-relative), task-level -- Release 5", "fig26a_cost_vs_share_R5.png",
           "Figure 26", "R5 raw")
for control, fname in [(COMPLEXITY_CONTROL, "fig26b_cost_vs_share_partial_R5.png"),
                       ("aei_education" if COMPLEXITY_CONTROL == "ridge" else "ridge", "fig26b_alt_aei_education_control_R5.png"
                        if COMPLEXITY_CONTROL == "ridge" else "fig26b_alt_ridge_control_R5.png")]:
    ps = partial_sample(5, control)
    cost_chart(ps, "x_resid", "y_resid", "Residual ln(R2R Cost Index)", "Residual ln(R2R Conversation Share)",
               "Task usage share vs. API cost index, controlling for subprocess, collaboration mode, and complexity -- Release 5",
               fname, "Figure 26", f"R5 partial, complexity control = {control}", through_origin=True)

[write] outputs/3_ch4_2_rq2_expertise/fig26a_cost_vs_share_R5.png


[write] outputs/3_ch4_2_rq2_expertise/fig26b_cost_vs_share_partial_R5.png


[write] outputs/3_ch4_2_rq2_expertise/fig26b_alt_aei_education_control_R5.png


## Fit statistics quoted in Section 4.2

In [24]:
rc.write_fit_log("rq2")

[write] outputs/3_ch4_2_rq2_expertise/fit_statistics.csv


,figure,panel,x,y,slope,intercept,r2,n,note
0,Figure 13,R5 gate ON,subprocess ridge expertise,R_o,-0.099473,1.651331e+00,0.457721,6,
1,Figure 13,R5 gate OFF,subprocess ridge expertise,R_o,-0.028790,7.973880e-01,0.026660,6,
2,Figure 14,R5 gate ON,subprocess ridge expertise,concentration,-0.258672,4.597111e+00,0.277152,6,
3,Figure 14,R5 gate OFF,subprocess ridge expertise,concentration,-0.142922,3.062641e+00,0.129600,6,
4,Figure 15,R5 gate ON,task ridge expertise,r_tilde,-0.014017,3.666836e-01,0.003587,91,
5,Figure 15,R5 gate OFF,task ridge expertise,r_tilde,0.014036,1.199844e-01,0.002688,91,
6,Figure 16,R5 gate ON,ridge_task_years,task dispersion,0.000377,2.596967e-03,0.001652,91,
7,Figure 16,R5 gate OFF,ridge_task_years,task dispersion,0.000661,-1.916504e-04,0.006889,91,
8,Figure 17,R5 gate ON,subprocess value,concentration,-0.008803,1.312368e+00,0.099590,6,
9,Figure 17,R5 gate OFF,subprocess value,concentration,-0.003345,1.170892e+00,0.022030,6,
